# Deploy strands-decider v19 on SageMaker AI and call it

This notebook walks through the deployment, in order:

1. **Stage the weights** (v19 checkpoint + Qwen3.5-2B base) in S3 with CodeBuild.
2. **Build the serving image** with CodeBuild and push it to ECR.
3. **Deploy** a SageMaker real-time endpoint on `ml.g6.xlarge`.
4. **Call it**: one question, then an 8-question request.
5. **Clean up.**

```
client ──► SageMaker endpoint (ml.g6.xlarge, 1x L4)
             └─ container: serve.py  (FastAPI: /ping, /invocations)
                  ├─ strands_decider engine (PyPI 0.1.0) + fla kernels
                  ├─ cross-request batcher (single-question requests)
                  └─ /opt/ml/model/{checkpoint,base}  ◄── S3 prefix, uncompressed
```

**Before you start**
- AWS credentials for the target account. Everything here runs in **us-east-2**.
- Quota for `ml.g6.xlarge for endpoint usage` (at least 1).
- `boto3` in this kernel. Nothing builds on your laptop: Docker and the model download both run in CodeBuild.
- Nothing else: the notebook writes the container source and buildspecs itself, into `build/` next to the notebook.

**Cost while running:** about \$1.13/h for the endpoint, plus a few cents of CodeBuild. Run the cleanup section when you're done.

## Prerequisites: permissions for the identity that runs this notebook

The cells below create AWS resources, so the identity running the notebook (your local AWS profile, or the SageMaker Studio execution role) needs these permissions. They are separate from the two roles that section 1 creates for CodeBuild and the endpoint.

| Service | Actions | Resource |
|---|---|---|
| IAM | `CreateRole`, `GetRole`, `PutRolePolicy`, `DeleteRolePolicy`, `DeleteRole`, `PassRole` (to CodeBuild and SageMaker) | `arn:aws:iam::<account>:role/strands-decider-*` |
| CodeBuild | `CreateProject`, `UpdateProject`, `StartBuild`, `BatchGetBuilds`, `DeleteProject` | `project/strands-decider-*` |
| ECR | `CreateRepository`, `DescribeRepositories`, `DescribeImages`, `DeleteRepository` | `repository/strands-decider` |
| S3 | `CreateBucket`, `ListBucket`; `GetObject`, `PutObject`, `DeleteObject` | `sagemaker-<region>-<account>`, prefix `strands-decider/` |
| SageMaker | Create, Describe and Delete for `Model`, `EndpointConfig` and `Endpoint`; `InvokeEndpoint` | `model/`, `endpoint-config/` and `endpoint/strands-decider-*` |
| CloudWatch | `GetMetricStatistics` | `*` |

**Running in SageMaker Studio.** A default Studio execution role usually already has the SageMaker, S3 (`sagemaker-*` buckets), ECR and CloudWatch permissions through `AmazonSageMakerFullAccess`. The usual gap is IAM: the role can't create roles, so section 1 fails with `AccessDenied ... iam:CreateRole`. From a machine with admin credentials, add this inline policy to the execution role (the role can't grant itself permissions). Replace `<account>`:

```json
{"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
  "Action": ["iam:CreateRole", "iam:GetRole", "iam:PutRolePolicy", "iam:DeleteRolePolicy", "iam:DeleteRole", "iam:PassRole"],
  "Resource": "arn:aws:iam::<account>:role/strands-decider-*"}]}
```

```bash
aws iam put-role-policy --role-name <studio-execution-role> \
  --policy-name strands-decider-iam --policy-document file://strands_decider_iam_min.json
```

If that returns `LimitExceeded`, the role's inline policies have reached their 10,240-character total. Either delete inline policies whose permissions are already granted elsewhere, or create the same document as a customer-managed policy and attach it with `aws iam attach-role-policy`. If CodeBuild or ECR calls fail later, add the CodeBuild and ECR rows above as well.

**Other Studio requirements:** internet access to PyPI (section 2 downloads the `strands-decider` wheel), and a writable working directory (`REPO = Path.cwd()` in section 0).


## 0. Configuration

In [ ]:
import json, time, zipfile, io
from pathlib import Path
import boto3

REGION = "us-east-2"
PREFIX = "strands-decider"                     # names roles, ECR repo, CodeBuild projects, S3 prefix
ENDPOINT = "strands-decider-v19"
INSTANCE = "ml.g6.xlarge"
IMAGE_TAG = "v19-0.1.0"

REPO = Path.cwd()
# The notebook writes the build sources here, one file per cell, then zips them.
BUILD_DIR = REPO / "build"
(BUILD_DIR / "container").mkdir(parents=True, exist_ok=True)
(BUILD_DIR / "weights").mkdir(parents=True, exist_ok=True)

# Pinned Hugging Face revisions
CKPT_REV = "bb282d786bc251fd4e3068de3ada9ddbb38127cd"   # StrandsAgents/strands-decider-2B-hobson-v19
BASE_REV = "b1485b2fa6dfa1287294f269f5fb618e03d52d7c"   # Qwen/Qwen3.5-2B-Base

session = boto3.Session(region_name=REGION)
ACCT = session.client("sts").get_caller_identity()["Account"]
BUCKET = f"sagemaker-{REGION}-{ACCT}"
MODEL_S3 = f"s3://{BUCKET}/{PREFIX}/model/v19/"

iam, s3, ecr = session.client("iam"), session.client("s3"), session.client("ecr")
cb, sm = session.client("codebuild"), session.client("sagemaker")
smr, cw = session.client("sagemaker-runtime"), session.client("cloudwatch")
print("account", ACCT, "| bucket", BUCKET, "| model prefix", MODEL_S3)

## 1. IAM roles and S3 bucket

Two least-privilege roles:
- **CodeBuild role:** writes its own logs, reads and writes `s3://<bucket>/strands-decider/*`, and pushes to the one ECR repo.
- **SageMaker execution role:** reads only the model prefix, pulls only that repo, and writes endpoint logs and metrics.


In [ ]:
def ensure_bucket():
    try:
        s3.head_bucket(Bucket=BUCKET)
    except s3.exceptions.ClientError:
        s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})
    print("bucket ok:", BUCKET)

def ensure_role(name, service, policy):
    trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
             "Principal": {"Service": service}, "Action": "sts:AssumeRole",
             "Condition": {"StringEquals": {"aws:SourceAccount": ACCT}}}]}
    try:
        iam.create_role(RoleName=name, AssumeRolePolicyDocument=json.dumps(trust))
    except iam.exceptions.EntityAlreadyExistsException:
        pass
    iam.put_role_policy(RoleName=name, PolicyName="inline", PolicyDocument=json.dumps(policy))
    return iam.get_role(RoleName=name)["Role"]["Arn"]

REPO_ARN = f"arn:aws:ecr:{REGION}:{ACCT}:repository/{PREFIX}"
CB_ROLE = ensure_role(f"{PREFIX}-codebuild-role", "codebuild.amazonaws.com", {"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
     "Resource": f"arn:aws:logs:{REGION}:{ACCT}:log-group:/aws/codebuild/{PREFIX}-*"},
    {"Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"],
     "Resource": f"arn:aws:s3:::{BUCKET}/{PREFIX}/*"},
    {"Effect": "Allow", "Action": "s3:ListBucket", "Resource": f"arn:aws:s3:::{BUCKET}",
     "Condition": {"StringLike": {"s3:prefix": f"{PREFIX}/*"}}},
    {"Effect": "Allow", "Action": "ecr:GetAuthorizationToken", "Resource": "*"},
    {"Effect": "Allow", "Action": ["ecr:BatchCheckLayerAvailability", "ecr:InitiateLayerUpload",
     "ecr:UploadLayerPart", "ecr:CompleteLayerUpload", "ecr:PutImage", "ecr:BatchGetImage",
     "ecr:GetDownloadUrlForLayer"], "Resource": REPO_ARN}]})
SM_ROLE = ensure_role(f"{PREFIX}-sagemaker-role", "sagemaker.amazonaws.com", {"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Action": "s3:GetObject", "Resource": f"arn:aws:s3:::{BUCKET}/{PREFIX}/model/*"},
    {"Effect": "Allow", "Action": "s3:ListBucket", "Resource": f"arn:aws:s3:::{BUCKET}",
     "Condition": {"StringLike": {"s3:prefix": f"{PREFIX}/model/*"}}},
    {"Effect": "Allow", "Action": "ecr:GetAuthorizationToken", "Resource": "*"},
    {"Effect": "Allow", "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer",
     "ecr:BatchCheckLayerAvailability"], "Resource": REPO_ARN},
    {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents",
     "logs:DescribeLogStreams"], "Resource": f"arn:aws:logs:{REGION}:{ACCT}:log-group:/aws/sagemaker/*"},
    {"Effect": "Allow", "Action": "cloudwatch:PutMetricData", "Resource": "*",
     "Condition": {"StringEquals": {"cloudwatch:namespace": "/aws/sagemaker/Endpoints"}}}]})
ensure_bucket()
print(CB_ROLE, SM_ROLE, sep="\n")
time.sleep(10)   # let IAM propagate before CodeBuild assumes the role

## 2. Build the container source, one file at a time

The serving image comes from four small files. The next cells write each one to `BUILD_DIR/container/`, so you can read exactly what goes into the image. Section 2.6 zips them for CodeBuild.

| File | Role |
|---|---|
| `Dockerfile` | Base image, CUDA PyTorch, the `strands-decider` package and its fused kernels |
| `serve.py` | The SageMaker HTTP contract: `/ping`, `/invocations`, model loading, warm-up |
| `batching.py` | Groups single-question requests from different callers into one GPU pass |
| `buildspec.yml` | CodeBuild steps: log in to ECR, `docker build`, `docker push` |

### What SageMaker expects from a custom ("bring your own") container

1. **Model files** appear at `/opt/ml/model`. Here that's the S3 prefix from section 3, copied uncompressed, with `checkpoint/` and `base/` folders.
2. SageMaker starts the container as **`docker run <image> serve`**. Our `ENTRYPOINT` is `python /app/serve.py`, so `serve` arrives as an ignored argument.
3. The container listens on **port 8080** and answers:
   - `GET /ping` with 200 once it's ready. SageMaker won't send traffic until it does, and gives up after `ContainerStartupHealthCheckTimeoutInSeconds`.
   - `POST /invocations` for each `invoke_endpoint` call, with the body passed through unchanged.
4. **Environment variables** from the SageMaker model's `Environment` are set in the container (`MODEL_NAME`, `MAX_BATCH`, ...).
5. stdout and stderr go to CloudWatch Logs at `/aws/sagemaker/Endpoints/<endpoint>`.

### 2.1 `Dockerfile`

- **`python:3.12-slim`** plus **`build-essential`**: Triton, which the fla kernels use, compiles a small C launcher the first time it runs, so the image needs a C compiler.
- **PyTorch 2.7.1 for CUDA 12.6** comes from PyTorch's own index. The PyPI build would be CPU-only or a different CUDA version.
- **`strands-decider==0.1.0`** is the engine and its FastAPI app. `flash-linear-attention` / `fla-core` are the fused kernels for the Qwen3.5 torso's linear-attention layers. Versions are pinned because `batching.py` calls private engine helpers.
- The build **checks that `fla` is installed but doesn't import it**. Importing it probes for a GPU driver, and CodeBuild has no GPU.
- **`HF_HUB_OFFLINE` / `TRANSFORMERS_OFFLINE`** mean the container never contacts Hugging Face. Everything loads from `/opt/ml/model`.

In [ ]:
%%writefile {BUILD_DIR}/container/Dockerfile
FROM python:3.12-slim

# Triton compiles a small C launcher at runtime, so the image needs a C compiler.
RUN apt-get update -qq \
 && apt-get install -y -qq --no-install-recommends build-essential \
 && rm -rf /var/lib/apt/lists/*

ENV PIP_NO_CACHE_DIR=1 PYTHONUNBUFFERED=1
RUN pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126
# Versions from strands-decider docs/inference.md + training/README.md (CUDA setup).
RUN pip install \
      strands-decider==0.1.0 \
      transformers==5.17.0 \
      peft==0.21.0 \
      flash-linear-attention==0.5.2 \
      fla-core==0.5.2 \
 && python -c "import importlib.util, torch, triton; assert importlib.util.find_spec('fla'), 'fla missing'; print('torch', torch.__version__, 'triton', triton.__version__)"
# Note: don't `import fla` at build time. It probes for a GPU driver and fails without one.

ENV HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 TRITON_CACHE_DIR=/tmp/triton
WORKDIR /app
COPY batching.py serve.py ./
EXPOSE 8080
# SageMaker runs the container with the argument "serve".
ENTRYPOINT ["python", "/app/serve.py"]

### 2.2 `serve.py`: what runs when the container starts

Everything happens at **import time, before the web server starts**. That's deliberate: `/ping` can't answer until the model is loaded and warmed up, so SageMaker sends no traffic to a container that isn't ready.

1. **Load the model.** `server.create_app(CHECKPOINT, device="cuda")` comes from the package. It loads the v19 adapter from `/opt/ml/model/checkpoint`. The checkpoint config's `base_model` was rewritten to `/opt/ml/model/base`, so the Qwen3.5-2B base loads from local files. The package's own routes come too (`/health`, `/v1/systemone`).
2. **Wrap the engine** in the `Batcher` (2.3), tuned by `MAX_BATCH`, `MAX_BATCH_TOKENS` and `MAX_DELAY_MS`.
3. **Add the SageMaker routes:**
   - `/ping` returns `{"status": "ok"}`.
   - `/invocations` validates the body as a `SystemOneRequest` (400 if invalid), tokenises it in a worker thread (`prepare`), queues it (`submit`) and waits for the GPU thread's answer. Caller errors, such as a choice whose options were cut off by truncation, return 422.
   - The custom attribute `unbatched` forces the engine's reference path. We used it to check that batched answers match.
4. **Warm up.** It runs one multi-question request, then single-question batches of 1, 2, 4, 8, 16 and 32 rows. The first pass compiles the fla Triton kernels, and new batch shapes can trigger more compiles. This takes minutes, which is why the endpoint's startup timeout is 900 s.
5. **Start the GPU thread** (`batcher.start()`).
6. **`uvicorn.run(..., workers=1)`** on `0.0.0.0:8080`. One worker means one copy of the model on the GPU and one thread that owns it.

The response adds `latency_ms` (time in the container, queue included) and `batch_size` (requests sharing the GPU pass) to the package's normal response.

In [ ]:
%%writefile {BUILD_DIR}/container/serve.py
"""SageMaker serving shim for strands-decider, with continuous batching.

Wraps the package's FastAPI app (strands_decider.server.create_app) and adds the SageMaker
contract: GET /ping and POST /invocations on 0.0.0.0:8080. /invocations takes and returns
the System One shapes of /v1/systemone, plus `latency_ms` (enqueue to answer) and
`batch_size` (rows in the GPU pass that served it).

Single-question requests from different callers are batched into one forward (see
batching.py). Multi-question requests keep the engine's shared-prefix path.

Tuning (environment variables):
  MAX_BATCH          rows per forward                    (default 32)
  MAX_BATCH_TOKENS   rows x padded length per forward    (default 16384)
  MAX_DELAY_MS       how long a short batch may wait     (default 5)

Send `X-Amzn-SageMaker-Custom-Attributes: unbatched` to force the engine's own
one-request path (used to check batched answers against the reference).
"""

from __future__ import annotations

import json
import logging
import os
import time

import uvicorn
from fastapi import Request
from fastapi.concurrency import run_in_threadpool
from fastapi.responses import JSONResponse
from pydantic import ValidationError

from batching import Batcher
from strands_decider import server
from strands_decider.schema import SystemOneRequest

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")
log = logging.getLogger("serve")

MODEL_DIR = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
CHECKPOINT = os.path.join(MODEL_DIR, "checkpoint")
MODEL_NAME = os.environ.get("MODEL_NAME", "strands-decider-2B-hobson-v19")

t0 = time.perf_counter()
app = server.create_app(CHECKPOINT, device="cuda", model_name=MODEL_NAME)
log.info("model loaded in %.1fs", time.perf_counter() - t0)

batcher = Batcher(
    server.get_engine(),
    max_batch=int(os.environ.get("MAX_BATCH", "32")),
    max_batch_tokens=int(os.environ.get("MAX_BATCH_TOKENS", "16384")),
    max_delay_ms=float(os.environ.get("MAX_DELAY_MS", "5")),
)


@app.get("/ping")
async def ping() -> dict:
    return {"status": "ok"}


@app.post("/invocations")
async def invocations(request: Request) -> JSONResponse:
    body = await request.body()
    try:
        req = SystemOneRequest.model_validate_json(body)
    except ValidationError as exc:
        return JSONResponse(status_code=400, content={"error": json.loads(exc.json())})
    attrs = request.headers.get("x-amzn-sagemaker-custom-attributes", "")
    try:
        job = await run_in_threadpool(batcher.prepare, req,
                                      force_unbatched="unbatched" in attrs)
        payload = await batcher.submit(job)
    except ValueError as exc:  # caller error, e.g. options truncated away
        return JSONResponse(status_code=422, content={"error": str(exc)})
    return JSONResponse(content=payload)


# Warm up before /ping answers: the first forwards compile the fla Triton kernels, and
# each new batch size can trigger another compile.
_WARM_STATE = "Help! My payouts have been failing for 3 days."
_WARM_Q = {"type": "choice", "instructions": "Route it.",
           "criteria": {"billing": "money", "technical": "bugs", "sales": "pricing"}}
_multi = SystemOneRequest.model_validate({"state": _WARM_STATE, "questions": {
    "a": _WARM_Q, "b": {"type": "noul", "instructions": "Does this convey urgency?"}}})
t = time.perf_counter()
batcher.run_unbatched(_multi)
log.info("warmup multi-question %.0f ms", (time.perf_counter() - t) * 1000)
for n in (1, 2, 4, 8, 16, batcher.max_batch):
    rows = [batcher.prepare(SystemOneRequest.model_validate(
        {"state": f"{_WARM_STATE} Ticket {i}.", "questions": {"q": _WARM_Q}})).row
        for i in range(n)]
    t = time.perf_counter()
    out = batcher.run_rows(rows)  # type: ignore[arg-type]
    log.info("warmup batch=%d %.0f ms: %s", n, (time.perf_counter() - t) * 1000,
             json.dumps(out[0]["answers"]))
batcher.start()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8080, workers=1, log_level="info")

Optional: see the package code `serve.py` builds on. This downloads the `strands-decider` wheel, with no install, and prints `create_app`. It shows where the checkpoint is loaded and the package's own `/v1/systemone` route, which `/invocations` replaces with a batched version.

In [ ]:
import subprocess, sys
pkg = BUILD_DIR / "pkg"
subprocess.run([sys.executable, "-m", "pip", "download", "strands-decider==0.1.0", "--no-deps",
                "-q", "-d", str(pkg)], check=True)
whl = next(pkg.glob("strands_decider-0.1.0-*.whl"))
text = zipfile.ZipFile(whl).read("strands_decider/server.py").decode()
print(text[text.index("def create_app"):text.index("def serve(")])

### 2.3 `batching.py`: sharing GPU passes between callers

The engine handles one request at a time. A request with **one question** is a single row (state + question). Running it alone uses only about 100 tokens of GPU work, which leaves the GPU mostly idle. The batcher builds those same rows for **different callers' requests** and runs them in one forward pass:

- **One GPU thread** waits for work. When a request arrives, it waits up to `MAX_DELAY_MS` for others to join. It then takes up to `MAX_BATCH` rows, as long as rows x padded length stays within `MAX_BATCH_TOKENS`. Under load the queue is never empty, so the delay never applies.
- **Requests with several questions** (like the 8-question routing request in section 5) skip batching. They run through the engine's own `evaluate`, which encodes the state once and shares it across the questions. That was already the efficient path, and batching it measured slower.
- **Rows are built with the engine's own helpers** (`_fit` for truncation, `_pad`, `_temperatures`, `_option_token_index`). A batched row is therefore exactly the tensor the unbatched path builds; only its neighbours in the pass differ.
- **Tokenising is serialised** with a lock, because the Rust tokenizer isn't thread-safe and `_fit` writes engine state. It takes about 1 ms.
- **Errors** fail only the requests in that pass, and the thread keeps serving.

Measured effect: about 3.6x throughput for single-question traffic.

In [ ]:
%%writefile {BUILD_DIR}/container/batching.py
"""Dynamic batching for strands-decider single-question requests.

The upstream engine runs one request at a time: one state, N questions. A request with a
single question takes the engine's plain batched path (each row = state + question).
This module builds those same rows across *different* requests and runs them in one
forward, so concurrent single-question requests share a GPU pass.

Scheduling: one worker thread owns the GPU. It sleeps until a request arrives, then
takes every waiting single-question request (up to MAX_BATCH rows and MAX_BATCH_TOKENS
padded tokens) and runs them together. MAX_DELAY_MS lets a short batch wait a little for
company; under load the queue is never empty and the delay never applies. Multi-question
requests keep the engine's shared-prefix path and run alone, in arrival order.

Row construction reuses the engine's own `_fit` (truncation), `_pad`, `_temperatures`
and `_option_token_index`, so a batched row is the same tensor input the unbatched path
builds; only the batch neighbours differ. Pinned to strands-decider==0.1.0 because it
calls those private helpers.
"""

from __future__ import annotations

import asyncio
import collections
import logging
import threading
import time
from dataclasses import dataclass, field
from typing import Any

import torch

from strands_decider.infer import SystemOneEngine, _option_token_index, _to_answer
from strands_decider.prompting import RenderedQuestion, render_question, render_state
from strands_decider.schema import SystemOneRequest, SystemOneResponse, Usage

log = logging.getLogger("batching")


@dataclass
class Row:
    """One single-question request, tokenised and ready to stack."""

    name: str
    rq: RenderedQuestion
    ids: list[int]
    opt: list[int] | None  # absolute option-token positions (pointer head), else None


@dataclass
class Job:
    request: SystemOneRequest
    row: Row | None  # None -> multi-question (or forced-unbatched) request
    loop: asyncio.AbstractEventLoop | None = None
    future: asyncio.Future | None = None
    enqueued: float = field(default_factory=time.perf_counter)


class Batcher:
    def __init__(self, engine: SystemOneEngine, *, max_batch: int = 32,
                 max_batch_tokens: int = 16384, max_delay_ms: float = 5.0) -> None:
        self.eng = engine
        self.max_batch = max_batch
        self.max_batch_tokens = max_batch_tokens
        self.max_delay = max_delay_ms / 1000
        self._queue: collections.deque[Job] = collections.deque()
        self._cond = threading.Condition()
        # `_fit` writes engine._last_offsets and the Rust tokenizer is not safe to share
        # across threads, so tokenisation is serialised (it is ~1 ms; the GPU is not).
        self._prep_lock = threading.Lock()
        self._pointer = engine.model.config.head_type == "pointer"
        self.stats = {"batches": 0, "rows": 0, "max_rows": 0}
        self._thread: threading.Thread | None = None

    # ---- request side ------------------------------------------------------

    def prepare(self, req: SystemOneRequest, *, force_unbatched: bool = False) -> Job:
        """Tokenise a request. Raises ValueError for caller errors, as the engine does."""
        if len(req.questions) != 1 or force_unbatched:
            return Job(req, None)
        name, q = next(iter(req.questions.items()))
        rq = render_question(q)
        cfg = self.eng.model.config
        if not self._pointer and rq.n_slots > cfg.num_slots:
            raise ValueError(
                f"question has {rq.n_slots} options but this model has {cfg.num_slots} slots"
            )
        state_text = render_state(req.state)
        with self._prep_lock:
            s, qs = self.eng._fit(state_text, [rq.text])
            offsets = self.eng._last_offsets[0]
        opt = None
        if self._pointer:
            # Same as engine._option_idx(rendered, base=len(s)), for one row.
            opt = [len(s) + i for i in _option_token_index(offsets, rq.option_spans, 0)]
        return Job(req, Row(name, rq, s + qs[0], opt))

    async def submit(self, job: Job) -> dict:
        job.loop = asyncio.get_running_loop()
        job.future = job.loop.create_future()
        with self._cond:
            self._queue.append(job)
            self._cond.notify()
        return await job.future

    # ---- GPU side ----------------------------------------------------------

    def start(self) -> None:
        self._thread = threading.Thread(target=self._loop, name="gpu-batcher", daemon=True)
        self._thread.start()

    def _take(self) -> list[Job]:
        with self._cond:
            while not self._queue:
                self._cond.wait()
            if self._queue[0].row is None:
                return [self._queue.popleft()]
            deadline = self._queue[0].enqueued + self.max_delay
            while len(self._queue) < self.max_batch:
                remaining = deadline - time.perf_counter()
                if remaining <= 0:
                    break
                self._cond.wait(remaining)
            batch: list[Job] = []
            width = 0
            while self._queue and len(batch) < self.max_batch:
                nxt = self._queue[0]
                if nxt.row is None:
                    break  # leave multi-question requests for their own turn
                w = max(width, len(nxt.row.ids))
                if batch and w * (len(batch) + 1) > self.max_batch_tokens:
                    break
                width = w
                batch.append(self._queue.popleft())
            return batch

    def _loop(self) -> None:
        while True:
            jobs = self._take()
            try:
                if jobs[0].row is None:
                    payloads = [self.run_unbatched(jobs[0].request)]
                else:
                    payloads = self.run_rows([j.row for j in jobs])  # type: ignore[misc]
                    self.stats["batches"] += 1
                    self.stats["rows"] += len(jobs)
                    self.stats["max_rows"] = max(self.stats["max_rows"], len(jobs))
                    if self.stats["batches"] % 200 == 0:
                        log.info("batches=%(batches)d rows=%(rows)d max_rows=%(max_rows)d",
                                 self.stats)
            except Exception as exc:  # fail every request in the batch, keep serving
                log.exception("batch of %d failed", len(jobs))
                for j in jobs:
                    self._resolve(j, exc)
                continue
            for j, p in zip(jobs, payloads, strict=True):
                p["batch_size"] = len(jobs)
                p["latency_ms"] = round((time.perf_counter() - j.enqueued) * 1000, 2)
                self._resolve(j, p)

    @staticmethod
    def _resolve(job: Job, value: Any) -> None:
        def set_() -> None:
            if job.future is None or job.future.done():
                return
            if isinstance(value, Exception):
                job.future.set_exception(value)
            else:
                job.future.set_result(value)

        assert job.loop is not None
        job.loop.call_soon_threadsafe(set_)

    def run_unbatched(self, req: SystemOneRequest) -> dict:
        return self.eng.evaluate(req).model_dump()

    @torch.inference_mode()
    def run_rows(self, rows: list[Row]) -> list[dict]:
        """One forward over rows from different requests -> one response dict per row."""
        eng = self.eng
        ids, mask = eng._pad([r.ids for r in rows])
        opt_idx = None
        if self._pointer:
            w = max(len(r.opt) for r in rows)  # type: ignore[arg-type]
            opt_idx = torch.tensor(
                [r.opt + [-1] * (w - len(r.opt)) for r in rows],  # type: ignore[operator]
                dtype=torch.long, device=eng.device,
            )
        out = eng.model(
            input_ids=ids,
            attention_mask=mask,
            n_slots=torch.tensor([r.rq.n_slots for r in rows], device=eng.device),
            temperature=eng._temperatures([r.rq.kind for r in rows]),
            opt_idx=opt_idx,
        )
        probs = out["log_probs"].exp().float().cpu()
        ntok = mask.sum(dim=1).tolist()
        smoothing = eng.model.config.ordinal_smoothing
        results = []
        for i, r in enumerate(rows):
            ans = _to_answer(r.rq, probs[i, : r.rq.n_slots].tolist(),
                             ordinal_smoothing=smoothing)
            results.append(SystemOneResponse(
                model=eng.cfg.model_name,
                answers={r.name: ans},
                usage=Usage(input_tokens=int(ntok[i]), output_tokens=1),
            ).model_dump())
        return results

### 2.4 `buildspec.yml`: how CodeBuild builds the image

It logs in to ECR, then builds and pushes. `--platform linux/amd64` matches the g6 hosts. `ECR_REPO_NAME` and `IMAGE_TAG` come from the CodeBuild project's environment (section 3). The CodeBuild project needs **privileged mode** to run Docker.

In [ ]:
%%writefile {BUILD_DIR}/container/buildspec.yml
version: 0.2
phases:
  pre_build:
    commands:
      - REGISTRY="$AWS_ACCOUNT_ID.dkr.ecr.$AWS_DEFAULT_REGION.amazonaws.com"
      - aws ecr get-login-password --region "$AWS_DEFAULT_REGION" | docker login --username AWS --password-stdin "$REGISTRY"
  build:
    commands:
      - docker build --platform linux/amd64 -t "${REGISTRY}/${ECR_REPO_NAME}:${IMAGE_TAG}" .
  post_build:
    commands:
      - docker push "${REGISTRY}/${ECR_REPO_NAME}:${IMAGE_TAG}"

### 2.5 Check the files

serve.py needs a Linux machine with an NVIDIA GPU and the model files laid out as SageMaker provides them, so this cell doesn't start it. It only checks that the Python files compile and that the Dockerfile copies the files you wrote.

In [ ]:
for f in ("serve.py", "batching.py"):   # syntax check without writing __pycache__
    compile((BUILD_DIR / "container" / f).read_text(), f, "exec")
dockerfile = (BUILD_DIR / "container/Dockerfile").read_text()
assert "COPY batching.py serve.py ./" in dockerfile and 'ENTRYPOINT ["python", "/app/serve.py"]' in dockerfile
for p in sorted((BUILD_DIR / "container").iterdir()):
    print(f"{p.name:<14} {p.stat().st_size:>6} bytes")

### 2.6 Zip and upload

CodeBuild reads its source from a zip in S3. The files must sit at the **root** of the zip, because CodeBuild looks for `buildspec.yml` and the Docker build context there.

In [ ]:
def zip_dir(d: Path, out: Path) -> Path:
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(d.iterdir()):
            if p.is_file():
                z.write(p, p.name)            # arcname = bare file name -> zip root
    return out

container_zip = zip_dir(BUILD_DIR / "container", BUILD_DIR / "container-src.zip")
print(container_zip, zipfile.ZipFile(container_zip).namelist())
s3.upload_file(str(container_zip), BUCKET, f"{PREFIX}/src/container-src.zip")
print("uploaded", f"s3://{BUCKET}/{PREFIX}/src/container-src.zip")

## 3. CodeBuild projects: weights and image

**Weights job** (`weights/buildspec.yml`):
- downloads the v19 checkpoint and the Qwen3.5-2B base at the pinned revisions;
- rewrites the checkpoint config's `base_model` to `/opt/ml/model/base`, so nothing is fetched at run time;
- syncs both to the model prefix **uncompressed**. SageMaker then copies the files directly with `CompressionType=None`, with no tar.gz to unpack at startup.

**Image job:** `docker build --platform linux/amd64` and a push to ECR. It needs `privilegedMode` for Docker-in-Docker. Building in CodeBuild means you don't need Docker locally, and the multi-GB image is pushed from inside AWS rather than over your own connection.

The weights job's buildspec, written the same way. It runs `snapshot_download` at the pinned revisions, rewrites `base_model`, then runs `aws s3 sync` to the model prefix.

In [ ]:
%%writefile {BUILD_DIR}/weights/buildspec.yml
# Stages pinned model weights to S3 as an uncompressed SageMaker model prefix.
# Layout at s3://$BUCKET/$PREFIX/:
#   checkpoint/  StrandsAgents/strands-decider-2B-hobson-v19 @ $CKPT_REV, config rewritten
#                so base_model points at /opt/ml/model/base (no Hub access at runtime)
#   base/        Qwen/Qwen3.5-2B-Base @ $BASE_REV (revision recorded in provenance.json)
version: 0.2
phases:
  install:
    commands:
      - pip install --quiet "huggingface_hub==1.5.0"
  build:
    commands:
      - |
        python3 - <<'PY'
        import json, os, shutil
        from huggingface_hub import snapshot_download
        ckpt = snapshot_download("StrandsAgents/strands-decider-2B-hobson-v19",
                                 revision=os.environ["CKPT_REV"], local_dir="/tmp/model/checkpoint")
        base = snapshot_download("Qwen/Qwen3.5-2B-Base",
                                 revision=os.environ["BASE_REV"], local_dir="/tmp/model/base")
        for d in ("/tmp/model/checkpoint/.cache", "/tmp/model/base/.cache"):
            shutil.rmtree(d, ignore_errors=True)
        # The published v19 uses the legacy config name.
        cfg_path = next(p for p in ("/tmp/model/checkpoint/strands_decider_config.json",
                                    "/tmp/model/checkpoint/hobson_config.json") if os.path.exists(p))
        shutil.copy(cfg_path, cfg_path + ".orig")
        cfg = json.load(open(cfg_path))
        assert cfg["base_model"] == "Qwen/Qwen3.5-2B-Base", cfg["base_model"]
        cfg["base_model"] = "/opt/ml/model/base"
        json.dump(cfg, open(cfg_path, "w"), indent=2)
        print("rewrote", cfg_path)
        PY
      - du -sh /tmp/model/*
      - aws s3 sync /tmp/model "s3://${BUCKET}/${PREFIX}/" --delete --only-show-errors
      - aws s3 ls --recursive "s3://${BUCKET}/${PREFIX}/" | head -50

In [ ]:
weights_zip = zip_dir(BUILD_DIR / "weights", BUILD_DIR / "weights-src.zip")
s3.upload_file(str(weights_zip), BUCKET, f"{PREFIX}/src/weights-src.zip")
print("uploaded", f"s3://{BUCKET}/{PREFIX}/src/weights-src.zip")

In [ ]:
try:
    ecr.create_repository(repositoryName=PREFIX, imageScanningConfiguration={"scanOnPush": True})
except ecr.exceptions.RepositoryAlreadyExistsException:
    pass

def ensure_project(name, zip_key, compute, env, privileged):
    spec = dict(name=name, source={"type": "S3", "location": f"{BUCKET}/{PREFIX}/src/{zip_key}"},
                artifacts={"type": "NO_ARTIFACTS"}, serviceRole=CB_ROLE, timeoutInMinutes=60,
                environment={"type": "LINUX_CONTAINER", "image": "aws/codebuild/standard:7.0",
                             "computeType": compute, "privilegedMode": privileged,
                             "environmentVariables": [{"name": k, "value": v, "type": "PLAINTEXT"}
                                                      for k, v in env.items()]})
    try:
        cb.create_project(**spec)
    except cb.exceptions.ResourceAlreadyExistsException:
        cb.update_project(**spec)

ensure_project(f"{PREFIX}-weights", "weights-src.zip", "BUILD_GENERAL1_MEDIUM",
               {"BUCKET": BUCKET, "PREFIX": f"{PREFIX}/model/v19", "CKPT_REV": CKPT_REV, "BASE_REV": BASE_REV},
               privileged=False)
ensure_project(f"{PREFIX}-image", "container-src.zip", "BUILD_GENERAL1_LARGE",
               {"AWS_ACCOUNT_ID": ACCT, "ECR_REPO_NAME": PREFIX, "IMAGE_TAG": IMAGE_TAG},
               privileged=True)
print("projects ready")

In [ ]:
def run_build(project):
    bid = cb.start_build(projectName=project)["build"]["id"]
    while True:
        b = cb.batch_get_builds(ids=[bid])["builds"][0]
        if b["buildStatus"] != "IN_PROGRESS":
            print(f"{project}: {b['buildStatus']} in {(b['endTime'] - b['startTime']).seconds}s")
            assert b["buildStatus"] == "SUCCEEDED", b.get("logs", {}).get("deepLink")
            return b
        time.sleep(20)

# The two jobs are independent; run them one after the other for clear logs.
run_build(f"{PREFIX}-weights")
run_build(f"{PREFIX}-image")

digest = ecr.describe_images(repositoryName=PREFIX, imageIds=[{"imageTag": IMAGE_TAG}])["imageDetails"][0]["imageDigest"]
IMAGE = f"{ACCT}.dkr.ecr.{REGION}.amazonaws.com/{PREFIX}@{digest}"   # pin by digest, not tag
print(IMAGE)
print(s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/model/v19/checkpoint/")["KeyCount"], "checkpoint files")

## 4. Deploy the endpoint

Settings that matter:
- **Image pinned by digest.** Re-tagging can't silently change what's deployed, and rollback is just pointing back at the old digest.
- **`ContainerStartupHealthCheckTimeoutInSeconds=900`.** `serve.py` warms up before `/ping` answers: the first forward passes compile the fla Triton kernels, which takes minutes. A short timeout would kill a healthy container.
- **`InferenceAmiVersion=al2-ami-sagemaker-inference-gpu-3-1`**, a recent NVIDIA driver for the CUDA 12.6 wheels.
- **Batcher settings** `MAX_BATCH=32`, `MAX_BATCH_TOKENS=16384`, `MAX_DELAY_MS=5` are the deployed values. They only affect single-question requests.

Creating the endpoint takes roughly 8-12 minutes.

In [ ]:
MODEL_NAME = f"{ENDPOINT}-batch"
sm.create_model(
    ModelName=MODEL_NAME, ExecutionRoleArn=SM_ROLE,
    PrimaryContainer={
        "Image": IMAGE, "Mode": "SingleModel",
        "ModelDataSource": {"S3DataSource": {"S3Uri": MODEL_S3, "S3DataType": "S3Prefix",
                                             "CompressionType": "None"}},
        "Environment": {"MODEL_NAME": "strands-decider-2B-hobson-v19",
                        "MAX_BATCH": "32", "MAX_BATCH_TOKENS": "16384", "MAX_DELAY_MS": "5"}})
sm.create_endpoint_config(
    EndpointConfigName=MODEL_NAME,
    ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": MODEL_NAME,
                         "InstanceType": INSTANCE, "InitialInstanceCount": 1,
                         "ModelDataDownloadTimeoutInSeconds": 1200,
                         "ContainerStartupHealthCheckTimeoutInSeconds": 900,
                         "InferenceAmiVersion": "al2-ami-sagemaker-inference-gpu-3-1"}])
sm.create_endpoint(EndpointName=ENDPOINT, EndpointConfigName=MODEL_NAME)

t0 = time.time()
sm.get_waiter("endpoint_in_service").wait(EndpointName=ENDPOINT, WaiterConfig={"Delay": 30, "MaxAttempts": 60})
print(f"InService after {(time.time() - t0) / 60:.1f} min")

If the endpoint fails, read the container log first. Failures before `/ping` answers are almost always a missing kernel dependency or the startup timeout:

In [ ]:
logs = session.client("logs")
group = f"/aws/sagemaker/Endpoints/{ENDPOINT}"
for st in logs.describe_log_streams(logGroupName=group, orderBy="LastEventTime", descending=True, limit=1)["logStreams"]:
    for e in logs.get_log_events(logGroupName=group, logStreamName=st["logStreamName"], limit=30)["events"]:
        print(e["message"].rstrip())

## 5. Call the endpoint

The request body uses the strands-decider "System One" shape: one `state` and named `questions`. Question types:
- **`noul`**: yes/no; returns a probability.
- **`score`**: a graded scale; returns an expected level, the probabilities per level and a confidence.
- **`choice`**: pick one option; returns the choice, the probabilities per option and a confidence.

Write questions so the obvious reading is the intended one. v19 tends to miss reversals ("which does NOT fit?").

In [ ]:
def ask(body, unbatched=False):
    kw = {"CustomAttributes": "unbatched"} if unbatched else {}
    r = smr.invoke_endpoint(EndpointName=ENDPOINT, ContentType="application/json",
                            Body=json.dumps(body), **kw)
    return json.loads(r["Body"].read())

one = ask({"state": {"request": "Translate 'good morning' into Spanish."},
           "questions": {"tier": {"type": "choice", "instructions": "How capable a model does `request` need?",
                                  "criteria": {"basic": "A quick, routine reply.",
                                               "standard": "Ordinary professional work needing some care.",
                                               "advanced": "Expert-level or high-stakes work."}}}})
print(json.dumps(one, indent=2))

### An 8-question routing request

This is the kind of request a prompt router sends: 6 yes/no questions, a 5-level difficulty score and a 7-way knowledge-area choice, all about the same state. The engine encodes the state once and shares it across the questions (the shared-prefix path), so 8 questions cost about 300 ms of GPU time, not 8 calls.

**Token budget:** about 882 input tokens per request. Roughly 93% is fixed question text; the user's request averages about 39 tokens. The model reads each question together with the state in a 4,096-token window, and a long state is trimmed from the end without an error. Send the user's prompt, not attached documents.

In [ ]:
example = {
  "state": {
    "request": "Our Kafka consumer group keeps rebalancing every ~5 minutes even though no consumers are crashing. Session timeout is 10s. What's going on?"
  },
  "questions": {
    "needs_reasoning": {
      "type": "noul",
      "instructions": "Does answering `request` require multi-step reasoning, such as a proof, a calculation, debugging, or weighing several factors to reach a conclusion?"
    },
    "writes_code": {
      "type": "noul",
      "instructions": "Does `request` ask for code to be written, changed, reviewed or debugged?"
    },
    "high_stakes": {
      "type": "noul",
      "instructions": "Could a wrong answer to `request` cause real harm: legal, medical, financial, security, or safety consequences?"
    },
    "long_input": {
      "type": "noul",
      "instructions": "Does `request` involve reading long material, such as many pages, several documents, a large codebase, or long transcripts?"
    },
    "medical": {
      "type": "noul",
      "instructions": "Is `request` about health, symptoms, medication or medical treatment?"
    },
    "has_image": {
      "type": "noul",
      "instructions": "Does `request` include or refer to an image, photo, screenshot, chart or diagram that must be looked at to answer it?"
    },
    "difficulty": {
      "type": "score",
      "instructions": "How difficult is `request` for a capable assistant?",
      "criteria": [
        "Trivial: a quick reply anyone could give.",
        "Easy: short, routine, little thought needed.",
        "Moderate: ordinary professional work needing some care.",
        "Hard: needs expertise or careful multi-step work.",
        "Very hard: expert-level, open-ended or novel problem."
      ]
    },
    "area": {
      "type": "choice",
      "instructions": "Which knowledge area does `request` belong to?",
      "criteria": {
        "coding": "Writing, changing, reviewing or debugging software, scripts or infrastructure.",
        "science_math": "Maths, physics, chemistry, biology or statistics problems that need working out or expert reasoning, or research analysis.",
        "medical": "Health, symptoms, medication, diagnosis or treatment.",
        "legal": "Law, contracts, legal rights, regulation or compliance.",
        "finance": "Money decisions: investing, tax, loans, budgets or financial analysis.",
        "business_writing": "Writing or reviewing documents for work: reports, plans, professional emails, presentations, marketing or analyses.",
        "general_knowledge": "Everyday questions and simple facts (including quick unit conversions and science trivia), casual chat, personal messages, rewording, translation or creative writing."
      }
    }
  }
}
print("questions:", list(example["questions"]))
resp = ask(example)
for k, v in resp["answers"].items():
    val = v.get("noul", v.get("score", v.get("choice")))
    print(f"{k:<16} {val if isinstance(val, str) else round(val, 3)}")
print("usage", resp["usage"], "| server latency", resp["latency_ms"], "ms")

## 6. Clean up

The endpoint is the only resource billed by the hour (about \$1.13/h for `ml.g6.xlarge`). Delete it as soon as you're done testing. Every step below ignores resources that are already gone, so you can rerun this section safely.

In [ ]:
from botocore.exceptions import ClientError

MODEL_NAME = f"{ENDPOINT}-batch"

def gone(fn, *args, **kwargs):
    """Run a delete call; treat 'not found' as already deleted."""
    try:
        fn(*args, **kwargs)
        return "deleted"
    except ClientError as e:
        err = e.response["Error"]
        # SageMaker reports a missing resource as ValidationException "Could not find ...".
        if err["Code"] in ("ResourceNotFoundException", "RepositoryNotFoundException", "NoSuchEntity") \
                or (err["Code"] == "ValidationException" and "could not find" in err.get("Message", "").lower()):
            return "not found"
        raise

# Stop the hourly charge first.
print("endpoint:", gone(sm.delete_endpoint, EndpointName=ENDPOINT))
try:
    sm.get_waiter("endpoint_deleted").wait(EndpointName=ENDPOINT, WaiterConfig={"Delay": 15, "MaxAttempts": 40})
except Exception:
    pass
print("endpoint config:", gone(sm.delete_endpoint_config, EndpointConfigName=MODEL_NAME))
print("model:", gone(sm.delete_model, ModelName=MODEL_NAME))

### Remove the build resources (optional)

These cost little or nothing while idle: the CodeBuild projects, the ECR repository and image, the weights in S3 (about 5 GB), the two IAM roles and the log groups. Keep them if you plan to redeploy, since rebuilding the image and restaging the weights takes about 15 minutes. Set `DELETE_ALL = True` to remove them.

In [ ]:
DELETE_ALL = False

if DELETE_ALL:
    for p in (f"{PREFIX}-weights", f"{PREFIX}-image"):
        print("codebuild", p, gone(cb.delete_project, name=p))
    print("ecr repository:", gone(ecr.delete_repository, repositoryName=PREFIX, force=True))

    n = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=f"{PREFIX}/"):
        keys = [{"Key": o["Key"]} for o in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET, Delete={"Objects": keys})
            n += len(keys)
    print(f"s3: deleted {n} objects under s3://{BUCKET}/{PREFIX}/ (the bucket itself is kept)")

    for role in (f"{PREFIX}-codebuild-role", f"{PREFIX}-sagemaker-role"):
        print("role policy", role, gone(iam.delete_role_policy, RoleName=role, PolicyName="inline"))
        print("role", role, gone(iam.delete_role, RoleName=role))

    logs = session.client("logs")
    for group in (f"/aws/sagemaker/Endpoints/{ENDPOINT}",
                  f"/aws/codebuild/{PREFIX}-weights", f"/aws/codebuild/{PREFIX}-image"):
        print("log group", group, gone(logs.delete_log_group, logGroupName=group))
    print("done")

If you added the `strands-decider-iam` inline policy to your Studio execution role for this notebook (see Prerequisites), remove it from a machine with admin credentials once you no longer need it.